[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CONNECTS-SCV/bio-guides/blob/main/ab%20db/antibody_db_guide/06_structure/06_structure_lab.ipynb)


# 06 — 구조예측 (IgFold 직접 실행)

> 본문: [`06_structure.md`](06_structure.md) 와 **한 절씩 짝지어** 보세요.
> **전 셀 실행 16초** (실측)

**이 노트북은 도구를 직접 돌립니다.** IgFold 를 **직접 돌려** Fv 구조를 예측하고(`my_run/`), 커밋된 예측 구조와 CA-RMSD 로 대조해요.
각 절은 **① 직접 실행 → ② 내 결과 확인 → ③ 레퍼런스 대조** 순서예요.
내가 만든 결과는 `my_run/` 에 쌓이고, 저장소에 커밋된 `data/` 는 **대조군(레퍼런스)** 으로만 씁니다.
어느 단계를 건너뛰거나 실패해도 `resolve()` 가 `data/` 로 폴백해서 뒤 절이 계속 돌아가요.

## 0) 부트스트랩 — 저장소 클론 · 도구 설치 · 작업 폴더 이동

**Colab**: 이 셀을 그대로 실행하면 클론 → 챕터 폴더 이동 → 필요한 도구 설치까지 한 번에 끝나요.
**로컬**: 챕터 폴더 안에서 열었다면 클론 없이 진행됩니다.

In [ ]:
# ====== Colab/로컬 공용 부트스트랩 (모든 챕터 공통) ======
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # 이 가이드 저장소 (fork 했다면 본인 주소로 바꾸세요)
CLONE_AS = "bio-guides"
CHAPTER  = "06_structure"
PIP_PKGS = "pandas matplotlib biopython igfold anarci abnumber"          # 이 챕터가 실제로 돌리는 도구 (pip 이름)
NEED_HMMER = True        # ANARCI 계열은 HMMER 의 hmmscan 실행파일이 필요해요 (pip 로는 안 깔려요)
PIN_TRANSFORMERS = "4.36.2"    # IgFold 체크포인트가 요구하는 transformers 버전(없으면 None)

import os, sys, subprocess, pathlib, shutil, importlib.util
IN_COLAB = "google.colab" in sys.modules

# HuggingFace 가중치 다운로드가 '멈춘 채' 매달리는 일을 막습니다.
# (멈춤은 예외가 안 나서 폴백이 안 걸립니다 — 타임아웃을 걸어 실패로 바꿔야 data/ 로 이어집니다)
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "30")   # 스트림 30초 무응답 → 끊고 재시도
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "15")

def _run(cmd):
    print("$", cmd); subprocess.run(cmd, shell=True, check=True)

_MARK = "antibody_viz.py"           # 이 파일이 있는 폴더가 가이드 루트

def _find_root():
    """가이드 루트를 찾는다 — 챕터 폴더 안, 루트, 클론된 저장소 어디서 열어도 동작."""
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    # 클론 직후: cwd '아래'만 깊이 3까지 훑는다.
    # (상위로 올라가 rglob 하면 Colab 에서는 / 전체를 뒤지게 된다 — 매우 느리고 엉뚱한 경로를 잡는다)
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "repo 루트를 못 찾았습니다. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

ADV_ROOT = ROOT.resolve()
os.chdir(ADV_ROOT / CHAPTER)        # 챕터 폴더로 이동 → data/·my_run/ 상대경로 동작
sys.path.insert(0, str(ADV_ROOT))   # antibody_viz import 보장
PY, SCRIPTS = sys.executable, ADV_ROOT / "scripts"

# --- 의존성 설치 -----------------------------------------------------------
_IMPORT = {"biopython": "Bio", "pyyaml": "yaml"}          # pip 이름 ≠ import 이름
def _have(pkg):
    mod = _IMPORT.get(pkg, pkg.split("==")[0])
    try:
        return importlib.util.find_spec(mod) is not None
    except Exception:
        return False

if NEED_HMMER and shutil.which("hmmscan") is None:
    # ANARCI 는 내부적으로 hmmscan 을 호출해요. pip install anarci 만으로는 안 깔려요.
    if IN_COLAB:
        _run("apt-get -qq update")                       # 인덱스가 낡으면 install 이 404 로 죽는다
        _run("apt-get -qq install -y hmmer")             # ← ANARCI 가 부르는 hmmscan
    else:
        print("[!] hmmscan 이 없어요 → conda install -c bioconda hmmer  (또는 apt install hmmer)")

_miss = [p for p in PIP_PKGS.split() if not _have(p)]
if _miss:
    _run(f'"{sys.executable}" -m pip -q install ' + " ".join(_miss))

if "igfold" in PIP_PKGS and importlib.util.find_spec("pkg_resources") is None:
    # setuptools 81+(2026-02) 이 pkg_resources 를 없앴는데 IgFold 의존성이 이걸 import 해요.
    _run(f'"{sys.executable}" -m pip -q install "setuptools<81"')
    importlib.invalidate_caches()

import glob as _glob
if IN_COLAB and not _glob.glob("/usr/share/fonts/**/*Nanum*", recursive=True):
    # Colab 에는 한글 폰트가 없어 그래프의 한글 라벨이 □ 로 깨집니다.
    _run("apt-get -qq update"); _run("apt-get -qq install -y fonts-nanum")


if PIN_TRANSFORMERS:
    # IgFold 체크포인트에는 옛 transformers 의 토크나이저 객체가 pickle 돼 있어요.
    # 최신 transformers(5.x) 로는 unpickle 이 실패해서(Trie/BasicTokenizer 없음) 버전을 맞춥니다.
    _ver = subprocess.run([sys.executable, "-c",
                           "import transformers;print(transformers.__version__)"],
                          capture_output=True, text=True).stdout.strip()
    if not _ver.startswith("4."):
        print(f"[transformers {_ver or 'none'} → {PIN_TRANSFORMERS}] IgFold 체크포인트 호환 버전으로 맞춥니다")
        _run(f'"{sys.executable}" -m pip -q install "transformers=={PIN_TRANSFORMERS}"')

# --- 내 산출물 폴더 & 폴백 규칙 --------------------------------------------
MYRUN = pathlib.Path("my_run"); MYRUN.mkdir(exist_ok=True)

def run_tool(args, timeout=1800):
    """도구를 서브프로세스로 실제 실행하고 출력을 셀에 그대로 보여줘요."""
    args = [str(a) for a in args]
    print("$", " ".join(args))
    try:
        p = subprocess.run(args, capture_output=True, text=True, timeout=timeout)
    except Exception as e:
        print(f"[실행 실패] {type(e).__name__}: {e}\n→ 이 단계는 건너뛰고 레퍼런스(data/)로 이어갑니다")
        return False
    out = ((p.stdout or "") + (p.stderr or "")).strip()
    print(out[-3000:] if out else "(출력 없음)")
    if p.returncode != 0:
        print(f"[실패] returncode={p.returncode} → 이 단계는 건너뛰고 레퍼런스(data/)로 이어갑니다")
    return p.returncode == 0

def resolve(name):
    """내가 방금 만든 my_run/ 결과를 우선 쓰고, 없으면 커밋된 data/ 로 폴백."""
    mine, ref = MYRUN/name, pathlib.Path("data")/name
    if mine.exists():
        print(f"[내 결과]   {mine}")
        return str(mine)
    print(f"[레퍼런스] {ref}   ← my_run 산출물이 없어 커밋본으로 이어갑니다")
    return str(ref)

print("ADV_ROOT :", ADV_ROOT)
print("작업 폴더 :", pathlib.Path.cwd(), "| Colab:", IN_COLAB)

## 1) 직접 실행 — IgFold 로 Fv 예측 (본문 6.1)

```bash
python scripts/run_igfold_demo.py --fasta data/demo_mab.fa --out my_run/demo_antibody_igfold.pdb
```
IgFold 는 항체 언어모델(AntiBERTy) + graph network 로 backbone 을 예측하고, **B-factor 컬럼에
잔기별 예측오차(Å)** 를 적어 줘요. 실행 중 나오는 함정 두 가지는 스크립트가 처리합니다.

- `torch ≥ 2.6` 의 `weights_only=True` → 체크포인트 로드 실패 → `weights_only=False` 로 감싸기
- 최신 `transformers`(5.x) → 체크포인트 unpickle 실패 → 부트스트랩이 **`transformers==4.36.2`** 로 맞춰 줌

> `RUN_IGFOLD = False` 로 두면 예측을 건너뛰고 커밋된 PDB(`data/`)로 나머지 절을 진행해요.

In [ ]:
RUN_IGFOLD = True     # 이 노트북에서 가장 무거운 단계(수십 초). False 면 커밋본으로 진행

if RUN_IGFOLD:
    ok = run_tool([PY, SCRIPTS/"run_igfold_demo.py",
                   "--fasta", "data/demo_mab.fa",
                   "--out", "my_run/demo_antibody_igfold.pdb"])
else:
    print("RUN_IGFOLD=False → 예측을 건너뛰고 레퍼런스 PDB 로 진행합니다.")

## 2) 내 결과 확인 — 사슬별 예측 신뢰도 (본문 6.2)

In [ ]:
pdb = resolve("demo_antibody_igfold.pdb")
ch = {}
for line in open(pdb):
    if line.startswith("ATOM") and line[12:16].strip() == "CA":
        ch.setdefault(line[21], []).append(float(line[60:66]))
for c2, v in sorted(ch.items()):
    print(f"chain {c2}: {len(v)} res | mean err {sum(v)/len(v):.2f} Å | max {max(v):.2f} Å")
print("\n[심화] Heavy 의 최대 예측오차 위치가 CDR-H3 예요 — 가장 다양하고 가장 불확실한 loop.")

## 3) 내 결과 확인 — 신뢰도 프로파일 그래프 (본문 6.2)

In [ ]:
from antibody_viz import structure_confidence
from IPython.display import Image
png = "my_run/06_structure_confidence.png"
structure_confidence(resolve("demo_antibody_igfold.pdb"),
                     "IgFold confidence — demo mAb (Fv, 내 실행 결과)", png)
Image(png)

## 4) 레퍼런스 대조 — 커밋된 예측 구조와 얼마나 같은가

같은 서열·같은 모델이라도 실행 환경(BLAS·스레드 수)에 따라 좌표가 소수점 단위로 흔들릴 수 있어요.
그래서 **CA 좌표 RMSD** 와 사슬별 예측오차 통계로 비교합니다. (RMSD ≈ 0 이면 재현 성공)

In [ ]:
import pathlib
mine_p = pathlib.Path("my_run/demo_antibody_igfold.pdb")
if mine_p.exists():
    from Bio.PDB import PDBParser, Superimposer
    p = PDBParser(QUIET=True)
    def ca(path):
        s = p.get_structure("x", path)
        return [a for a in s.get_atoms() if a.get_id() == "CA"]
    a, b = ca(mine_p), ca("data/demo_antibody_igfold.pdb")
    print(f"CA 원자 수 — 내 결과 {len(a)} / 레퍼런스 {len(b)}")
    if len(a) == len(b):
        sup = Superimposer(); sup.set_atoms(b, a)
        print(f"CA-RMSD (내 예측 vs 커밋 예측) = {sup.rms:.3f} Å")
    for label, path in [("내 결과", mine_p), ("레퍼런스", "data/demo_antibody_igfold.pdb")]:
        d = {}
        for line in open(path):
            if line.startswith("ATOM") and line[12:16].strip() == "CA":
                d.setdefault(line[21], []).append(float(line[60:66]))
        print(label, {k: f"mean {sum(v)/len(v):.2f} / max {max(v):.2f} Å" for k, v in sorted(d.items())})
else:
    print("my_run 예측이 없어 대조를 건너뜁니다 (RUN_IGFOLD=False 였거나 실행 실패).")

## 5) 3D 구조 렌더 — 예측오차 컬러링 (본문 6.2)

색은 위 그래프와 같은 잔기별 예측오차(B-factor): **파랑=신뢰 / 빨강=불확실** → 빨간 loop 가 CDR-H3.

> **PyMOL 은 pip 로 설치되지 않아요**(Colab 미지원). 그래서 이 절만은 **저장소에 커밋된 렌더 이미지**를
> 그대로 보여줍니다. 로컬에 open-source PyMOL 이 있으면 자동으로 다시 렌더해요.

In [ ]:
import shutil, subprocess
from IPython.display import Image
png = "06_structure_3d.png"
if shutil.which("pymol"):
    try:
        subprocess.run(["pymol", "-cq", str(ADV_ROOT/"scripts"/"render_06_structure.pml")],
                       cwd=str(ADV_ROOT), check=True, capture_output=True, text=True, timeout=180)
        print("PyMOL 재렌더 완료 →", png)
    except Exception as e:
        print("PyMOL 실행 실패 → 커밋된 렌더 표시:", type(e).__name__)
else:
    print("PyMOL 없음(예: Colab) → 커밋된 렌더를 표시합니다:", png)
Image(png)

> 다음 → 본문 [07. interface 분석](../07_interface/07_interface.md)